# SaaSビジネスメトリクス分析

サブスクリプション型SaaSの月次収益（MRR）・解約率（Churn）・LTV を分析します。

---

**実行環境:** MySQL 8.0 / Python 3 / pandas  
**DB:** `sql_portfolio`（`SETUP.md` の手順で事前に構築）

## セットアップ

In [1]:
import mysql.connector
import pandas as pd
from IPython.display import display, HTML

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.set_option('display.float_format', '{:,.2f}'.format)

con = mysql.connector.connect(
    host='localhost', user='root', password='',
    database='sql_portfolio',
    charset='utf8mb4'
)

def run(sql):
    return pd.read_sql(sql, con)


---

## 分析クエリ

### 01. 月次MRR推移

**ビジネス課題:** 月ごとの総MRRと前月比を追跡し、事業の成長トレンドを把握する。

**使用テクニック:** `SUM() OVER` (累積), `LAG()`

In [2]:
sql = '''
WITH monthly_changes AS (
    SELECT
        DATE_FORMAT(month, '%Y-%m') AS ym,
        SUM(mrr)                    AS net_mrr_change
    FROM saas_mrr_history
    GROUP BY DATE_FORMAT(month, '%Y-%m')
),
cumulative AS (
    SELECT
        ym,
        net_mrr_change,
        SUM(net_mrr_change) OVER (ORDER BY ym) AS total_mrr
    FROM monthly_changes
)
SELECT
    ym,
    net_mrr_change,
    total_mrr,
    LAG(total_mrr) OVER (ORDER BY ym) AS prev_mrr,
    ROUND(
        (total_mrr - LAG(total_mrr) OVER (ORDER BY ym)) /
        NULLIF(LAG(total_mrr) OVER (ORDER BY ym), 0) * 100, 1
    ) AS mom_growth_pct
FROM cumulative
ORDER BY ym;
'''
df = run(sql)
display(df)

C:\Users\willi\AppData\Local\Temp\ipykernel_2916\2420247800.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, con)


,ym,net_mrr_change,total_mrr,prev_mrr,mom_growth_pct
0,2023-04,"50,000.00","50,000.00",NaN,NaN
1,2023-05,"80,000.00","130,000.00","50,000.00",160.00
2,2023-06,"15,000.00","145,000.00","130,000.00",11.50
3,2023-07,"15,000.00","160,000.00","145,000.00",10.30
4,2023-08,"85,000.00","245,000.00","160,000.00",53.10
5,2023-09,"30,000.00","275,000.00","245,000.00",12.20
6,2023-10,"60,000.00","335,000.00","275,000.00",21.80
7,2023-11,"40,000.00","375,000.00","335,000.00",11.90
8,2023-12,"85,000.00","460,000.00","375,000.00",22.70
9,2024-01,"95,000.00","555,000.00","460,000.00",20.70


### 02. MRR変動の内訳（新規/拡張/縮小/解約）

**ビジネス課題:** MRRの増減要因を分類し可視化する。

**使用テクニック:** 条件付き集計 `CASE + SUM`

In [3]:
sql = '''
SELECT
    DATE_FORMAT(month, '%Y-%m') AS ym,
    SUM(CASE WHEN change_type = 'new'          THEN mrr ELSE 0 END) AS new_mrr,
    SUM(CASE WHEN change_type = 'expansion'    THEN mrr ELSE 0 END) AS expansion_mrr,
    SUM(CASE WHEN change_type = 'contraction'  THEN mrr ELSE 0 END) AS contraction_mrr,
    SUM(CASE WHEN change_type = 'churn'        THEN mrr ELSE 0 END) AS churn_mrr,
    SUM(CASE WHEN change_type = 'reactivation' THEN mrr ELSE 0 END) AS reactivation_mrr,
    SUM(mrr)                                                         AS net_change
FROM saas_mrr_history
GROUP BY DATE_FORMAT(month, '%Y-%m')
ORDER BY ym;
'''
df = run(sql)
display(df)

C:\Users\willi\AppData\Local\Temp\ipykernel_2916\2420247800.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, con)


,ym,new_mrr,expansion_mrr,contraction_mrr,churn_mrr,reactivation_mrr,net_change
0,2023-04,"50,000.00",0.00,0.00,0.00,0.00,"50,000.00"
1,2023-05,"80,000.00",0.00,0.00,0.00,0.00,"80,000.00"
2,2023-06,"15,000.00",0.00,0.00,0.00,0.00,"15,000.00"
3,2023-07,"25,000.00",0.00,"-10,000.00",0.00,0.00,"15,000.00"
4,2023-08,"55,000.00","45,000.00",0.00,"-15,000.00",0.00,"85,000.00"
5,2023-09,"20,000.00","10,000.00",0.00,0.00,0.00,"30,000.00"
6,2023-10,"60,000.00",0.00,0.00,0.00,0.00,"60,000.00"
7,2023-11,"30,000.00","10,000.00",0.00,0.00,0.00,"40,000.00"
8,2023-12,"75,000.00","35,000.00","-10,000.00","-15,000.00",0.00,"85,000.00"
9,2024-01,"110,000.00","80,000.00","-45,000.00","-50,000.00",0.00,"95,000.00"


### 03. 月次チャーンレート

**ビジネス課題:** 月ごとの解約率を算出し、顧客維持の健全性を評価する。

**使用テクニック:** CTE, サブクエリ

In [4]:
sql = '''
WITH monthly_signups AS (
    -- 各月末時点での累計契約数
    SELECT
        DATE_FORMAT(signup_date, '%Y-%m') AS ym,
        COUNT(*)                           AS new_signups
    FROM saas_customers
    GROUP BY DATE_FORMAT(signup_date, '%Y-%m')
),
monthly_churns AS (
    SELECT
        DATE_FORMAT(churned_date, '%Y-%m') AS ym,
        COUNT(*)                            AS churned_count
    FROM saas_customers
    WHERE churned_date IS NOT NULL
    GROUP BY DATE_FORMAT(churned_date, '%Y-%m')
),
-- 各月の顧客数を累積計算
months AS (
    SELECT DISTINCT DATE_FORMAT(month, '%Y-%m') AS ym
    FROM saas_mrr_history
),
customer_counts AS (
    SELECT
        m.ym,
        (SELECT COUNT(*) FROM saas_customers
         WHERE DATE_FORMAT(signup_date, '%Y-%m') <= m.ym
           AND (churned_date IS NULL OR DATE_FORMAT(churned_date, '%Y-%m') > m.ym)
        ) AS active_customers_eom,
        COALESCE(mc.churned_count, 0) AS churned_in_month
    FROM months m
    LEFT JOIN monthly_churns mc ON m.ym = mc.ym
)
SELECT
    ym,
    active_customers_eom,
    churned_in_month,
    ROUND(
        churned_in_month /
        NULLIF(active_customers_eom + churned_in_month, 0) * 100, 2
    ) AS churn_rate_pct
FROM customer_counts
ORDER BY ym;
'''
df = run(sql)
display(df)

C:\Users\willi\AppData\Local\Temp\ipykernel_2916\2420247800.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, con)


,ym,active_customers_eom,churned_in_month,churn_rate_pct
0,2023-04,1,0,0.00
1,2023-05,4,0,0.00
2,2023-06,5,0,0.00
3,2023-07,8,0,0.00
4,2023-08,9,1,10.00
5,2023-09,11,0,0.00
6,2023-10,15,0,0.00
7,2023-11,17,0,0.00
8,2023-12,20,1,4.76
9,2024-01,26,1,3.70


### 04. 顧客LTV推計

**ビジネス課題:** ARPU÷チャーンレートで理論LTVを推計する。

**使用テクニック:** `CROSS JOIN`, `NULLIF`, `TIMESTAMPDIFF`

In [5]:
sql = '''
WITH arpu AS (
    -- 直近月のARPU（アクティブ顧客のみ）
    SELECT
        ROUND(AVG(p.monthly_price), 0) AS avg_mrr_per_customer
    FROM saas_customers sc
    JOIN saas_plans p ON sc.plan_id = p.plan_id
    WHERE sc.is_active = 1
),
churn AS (
    -- 全期間の月次平均チャーンレート
    SELECT
        ROUND(
            COUNT(CASE WHEN churned_date IS NOT NULL THEN 1 END) /
            NULLIF(COUNT(*), 0) /
            -- 平均在籍月数で割って月次レートに変換
            NULLIF(
                TIMESTAMPDIFF(MONTH,
                    (SELECT MIN(signup_date) FROM saas_customers),
                    CURDATE()
                ), 0
            ) * 100, 2
        ) AS monthly_churn_rate_pct
    FROM saas_customers
)
SELECT
    a.avg_mrr_per_customer              AS arpu,
    c.monthly_churn_rate_pct            AS monthly_churn_pct,
    CASE
        WHEN c.monthly_churn_rate_pct > 0 THEN
            ROUND(a.avg_mrr_per_customer / (c.monthly_churn_rate_pct / 100), 0)
        ELSE NULL
    END                                 AS estimated_ltv,
    CASE
        WHEN c.monthly_churn_rate_pct > 0 THEN
            ROUND(1 / (c.monthly_churn_rate_pct / 100), 1)
        ELSE NULL
    END                                 AS avg_lifetime_months
FROM arpu a
CROSS JOIN churn c;
'''
df = run(sql)
display(df)

C:\Users\willi\AppData\Local\Temp\ipykernel_2916\2420247800.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, con)


,arpu,monthly_churn_pct,estimated_ltv,avg_lifetime_months
0,"19,655.00",0.79,"2,487,975.00",126.60


### 05. プラン別ARPU・解約率比較

**ビジネス課題:** プラン別の収益性とリスクを比較する。

**使用テクニック:** `LEFT JOIN`, `GROUP BY`, 条件付き集計

In [6]:
sql = '''
SELECT
    p.plan_id,
    p.plan_name,
    p.monthly_price,
    COUNT(sc.customer_id)                                         AS total_customers,
    SUM(CASE WHEN sc.is_active = 1 THEN 1 ELSE 0 END)            AS active_customers,
    SUM(CASE WHEN sc.churned_date IS NOT NULL THEN 1 ELSE 0 END) AS churned_customers,
    ROUND(
        SUM(CASE WHEN sc.churned_date IS NOT NULL THEN 1 ELSE 0 END) /
        NULLIF(COUNT(sc.customer_id), 0) * 100, 1
    )                                                              AS churn_rate_pct,
    ROUND(
        SUM(CASE WHEN sc.is_active = 1 THEN p.monthly_price ELSE 0 END), 0
    )                                                              AS current_total_mrr,
    ROUND(
        AVG(CASE WHEN sc.is_active = 1 THEN p.monthly_price END), 0
    )                                                              AS arpu
FROM saas_plans p
LEFT JOIN saas_customers sc ON p.plan_id = sc.plan_id
GROUP BY p.plan_id, p.plan_name, p.monthly_price
ORDER BY p.monthly_price DESC;
'''
df = run(sql)
display(df)

C:\Users\willi\AppData\Local\Temp\ipykernel_2916\2420247800.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, con)


,plan_id,plan_name,monthly_price,total_customers,active_customers,churned_customers,churn_rate_pct,current_total_mrr,arpu
0,3,Enterprise,"50,000.00",8,7.00,1.00,12.50,"350,000.00","50,000.00"
1,2,Professional,"15,000.00",18,11.00,7.00,38.90,"165,000.00","15,000.00"
2,1,Starter,"5,000.00",14,11.00,3.00,21.40,"55,000.00","5,000.00"


---

In [7]:
con.close()
print('接続を閉じました。')

接続を閉じました。
